<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/banking77-intent-classification/notebooks/pretrained_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
!git clone https://github.com/TheFaridKarami/farid-ai-lab.git

fatal: destination path 'farid-ai-lab' already exists and is not an empty directory.


In [61]:
import torch
import numpy as np
import pandas as pd
import transformers
import datasets
import sklearn
from pathlib import Path

In [62]:
PROJECT_ROOT = (Path("/content/farid-ai-lab")
 / "projects/nlp/banking77-intent-classification")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

In [63]:
print(PROJECT_ROOT.exists())
print(DATA_DIR.exists())
print(TRAIN_PATH.exists())
print(TEST_PATH.exists())

True
True
True
True


In [64]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

In [65]:
print(train_df.describe())

                                  text                     label
count                            10003                     10003
unique                           10003                        77
top     I am still waiting on my card?  card_payment_fee_charged
freq                                 1                       187


In [66]:
print(train_df.dtypes)

text     object
label    object
dtype: object


In [67]:
print(train_df.head())

                                                text         label
0                     I am still waiting on my card?  card_arrival
1  What can I do if my card still hasn't arrived ...  card_arrival
2  I have been waiting over a week. Is the card s...  card_arrival
3  Can I track my card while it is in the process...  card_arrival
4  How do I know if I will get my card, or if it ...  card_arrival


In [68]:
print(train_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10003 entries, 0 to 10002
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    10003 non-null  object
 1   label   10003 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB
None


In [69]:
print(train_df.isnull().sum())

text     0
label    0
dtype: int64


In [70]:
print(train_df.columns.to_list())

['text', 'label']


In [71]:
print(train_df["label"].nunique())

77


In [72]:
print(train_df.value_counts().describe())

count    10003.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: count, dtype: float64


In [73]:
print(train_df["label"].isna().sum())

0


In [74]:
print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("Training classes:", train_df["label"].nunique())
print("Testing classes:", test_df["label"].nunique())

print("\nTraining class distribution:")
print(train_df["label"].value_counts().sort_values())

print("\nClasses missing from test set:")
print(set(train_df["label"]) - set(test_df["label"]))

Training samples: 10003
Testing samples: 3080
Training classes: 77
Testing classes: 77

Training class distribution:
label
contactless_not_working                              35
virtual_card_not_working                             41
card_acceptance                                      59
card_swallowed                                       61
lost_or_stolen_card                                  82
                                                   ... 
cash_withdrawal_charge                              177
wrong_amount_of_cash_received                       180
balance_not_updated_after_cheque_or_cash_deposit    181
direct_debit_payment_not_recognised                 182
card_payment_fee_charged                            187
Name: count, Length: 77, dtype: int64

Classes missing from test set:
set()


In [78]:
print(len(train_df))
print(len(test_df))
print("--" * 20)
print(train_df["label"].nunique())
print(test_df["label"].nunique())

10003
3080
----------------------------------------
77
77


In [82]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels= 77
)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [83]:
print(model)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [84]:
print("Architecture:", model.config.architectures)
print("Number of labels:", model.config.num_labels)
print("Model parameters:", sum(p.numel() for p in model.parameters()))

Architecture: ['DistilBertForMaskedLM']
Number of labels: 77
Model parameters: 67012685


In [87]:
print(sum(p.numel() for p in model.parameters()))

67012685
